# 00 — Prepare the backdoor-defense study data

This notebook is the first step in the tutorial.

We study a dummy **backdoor defense** in this tutorial. The specific technique is not important. 

## Causal question

The causal question is:

> **To what extent does applying the backdoor defense, instead of baseline random filtering, cause the probability of successful backdoor detection to change?**

In this tutorial:

| Variable | Meaning |
|---|---|
| `treatment = 0` | baseline **random filtering** |
| `treatment = 1` | **backdoor defense** applied |
| `outcome = 0` | backdoor detection failed |
| `outcome = 1` | backdoor detection succeeded |

Because `outcome` is binary, its mean is the **detection success rate (DSR)**.

### Tutorial path

**00 Data preparation → 01 Correlational analysis → 02 Causal inference**


## 1. What is synthetic and what comes from the source data?

The raw CSV contains real code/docstring examples used as heterogeneous software units.

The tutorial adds a **synthetic observational backdoor-defense study**. The treatment assignment and detection outcome are generated so that the true causal structure is known.

Please note that the causal treatment/outcome are synthetic and are **not presented as empirical backdoor-defense results**;

Each row in the final causal table represents one code example observed under **one** treatment condition with **one** detection outcome.


## 2. Configure the preparation

The parameters below define the files and settings used in this notebook.


In [ ]:
from src.causal_data_prep import (
    DEFAULT_COVARIATES,
    engineer_features,
    load_source_data,
    make_synthetic_observational_data,
    save_causal_dataset,
    validate_causal_dataset,
)

def default_params():
    return {
        "source_dataset": "data/raw_code.csv",
        "lizard_cache_folder": "cache/lizard",
        "causal_dataset": "data/causal_data.csv",
        "random_seed": 42,
        "covariate_columns": DEFAULT_COVARIATES,
    }

params = default_params()
params

ImportError: cannot import name 'save_ground_truth_dag' from 'src.causal_data_prep' (/workspaces/ci4sesci/notebooks/bowen_class/src/causal_data_prep.py)

## 3. Load the source examples

The source file contains:

- `input_code`: the code example;
- `output_docstring`: its associated docstring;
- `reviewer_experience`: a synthetic pre-treatment context variable;
- `rollout_eligibility`: a synthetic indicator of whether the example is eligible for the defense rollout;
- `noise_feature`: a synthetic background variable with no intended meaning.

The last three fields exist only to make the causal exercise richer. They are known **before treatment**.


In [ ]:
source_df = load_source_data(params["source_dataset"])

print(f"Loaded {len(source_df):,} source examples.")
source_df[
    [
        "input_code",
        "output_docstring",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].head()


## 4. Engineer baseline code features

We derive a small set of readable code characteristics:

- `code_number_tokens`
- `code_complexity`
- `code_num_identifiers`
- `code_num_strings`

These are measurements. Their causal roles are **not** determined merely by being available in the dataset.


In [ ]:
feature_df = engineer_features(
    source_df,
    cache_dir=params["lizard_cache_folder"],
)

feature_df[
    [
        "code_number_tokens",
        "code_complexity",
        "code_num_identifiers",
        "code_num_strings",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].describe().T


## 5. Generate one observed treatment and one observed outcome

The synthetic data assigns each unit to exactly one observed condition:

- **0 — Random filtering:** the baseline/control condition;
- **1 — Backdoor defense:** the backdoor defense is applied.

The generated `outcome` records whether backdoor detection succeeded for that unit.

The data also contains a few variables measured after treatment. They make the later DAG exercise more realistic because strong association does not necessarily mean “confounder.”

The generator internally knows both potential detection probabilities for the synthetic data, but the exported dataset contains only the treatment and outcome actually observed for each unit.


In [ ]:
causal_df, study_info = make_synthetic_observational_data(
    feature_df,
    seed=params["random_seed"],
)

print(f"Backdoor-defense prevalence: {study_info.treatment_prevalence:.3f}")
print(f"Observed detection success rate: {study_info.outcome_prevalence:.3f}")

causal_df.head()


## 6. Validate the causal table

Before moving on, check that:

- every unit appears once;
- both treatment conditions are present;
- `outcome` is binary;
- the analysis variables are numeric and finite;
- there are no missing analysis values.


In [ ]:
validation_summary = validate_causal_dataset(
    causal_df,
    covariates=params["covariate_columns"],
)

validation_summary


## 7. Save the handoff to notebook 01

`data/causal_data.csv` is the dataset used in notebooks 01 and 02.

The notebook also writes hidden synthetic-study information used only in the **reveal** section of notebook 02.

If you want to do the exercise yourself, do not inspect the ground-truth files yet.


In [ ]:
data_path = save_causal_dataset(
    causal_df,
    params["causal_dataset"],
)
truth_path = save_ground_truth_dag(
    params["ground_truth_dag"],
)
metadata_path = save_study_metadata(
    study_info,
    params["study_metadata"],
)

print(f"Saved causal dataset: {data_path}")
print(f"Saved hidden DAG: {truth_path}")
print(f"Saved hidden study metadata: {metadata_path}")


## Transition to notebook 01

Notebook 00 answers:

> **What was observed for each unit?**

Notebook 01 will deliberately ignore the hidden generating DAG and ask:

> **What relationships can we see in the observed data?**

That is a correlational question, not yet a causal one.
